# 00 - Infrastructure Setup

**Purpose:** One-time setup of database, schemas, stages, and governance tables.

**Run:** Once at project initialization, or when infrastructure changes needed.

**Runtime:** ~2 minutes

In [1]:
# 1. Setup & Imports
from snowflake.snowpark import Session
from utils import get_connection_config

conn_cfg = get_connection_config()
DB_NAME = conn_cfg["database"]
SCHEMA_DATA = conn_cfg["schema_data"]
STAGE_ARTIFACTS = conn_cfg["stage_artifacts"]
COMPUTE_POOL = conn_cfg["compute_pool"]
WAREHOUSE = conn_cfg["warehouse"]
TARGET_INSTANCES = conn_cfg["target_cluster_size"]

session = Session.builder.getOrCreate()
print(f"Connected to account: {session.get_current_account()}")
print(f"Current role: {session.get_current_role()}")

from modin.config import AutoSwitchBackend; AutoSwitchBackend.disable()


Connected to account: "SFSENORTHAMERICA-AFERAS_AWS1"
Current role: "ML_DEV_ROLE"


In [2]:
# 2. Configuration (loaded from config.yaml)
print(f"DB_NAME: {DB_NAME}")
print(f"SCHEMA_DATA: {SCHEMA_DATA}")
print(f"STAGE_ARTIFACTS: {STAGE_ARTIFACTS}")
print(f"COMPUTE_POOL: {COMPUTE_POOL}")

DB_NAME: MMT_POC
SCHEMA_DATA: FORECASTING
STAGE_ARTIFACTS: ML_STAGE
COMPUTE_POOL: MMT_POC_POOL_50


In [3]:
# 4. Create Database and Schemas
session.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}").collect()
session.sql(f"USE DATABASE {DB_NAME}").collect()

session.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DATA}").collect()
session.sql(f"USE SCHEMA {SCHEMA_DATA}").collect()

print(f"✓ Database {DB_NAME} ready")
print(f"✓ Schema {SCHEMA_DATA} ready")

✓ Database MMT_POC ready
✓ Schema FORECASTING ready


In [4]:
# 4. Create Stages
session.sql(f"""CREATE STAGE IF NOT EXISTS {STAGE_ARTIFACTS}
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')""").collect()

print(f"✓ Stage @{STAGE_ARTIFACTS} ready")

✓ Stage @ML_STAGE ready


In [5]:
session.sql("""
CREATE TABLE IF NOT EXISTS FEATURE_TABLE (
    ACTUAL_DATE DATE,
    PARTITION_ID VARCHAR(10),
    FEATURE_1 FLOAT,
    FEATURE_2 FLOAT,
    FEATURE_3 FLOAT,
    FEATURE_4 FLOAT,
    FEATURE_5 FLOAT,
    TARGET FLOAT
)
""").collect()

print("✓ Dimension and fact tables ready")

✓ Dimension and fact tables ready


In [6]:
# 7. Create Model Governance Tables
session.sql("""
CREATE TABLE IF NOT EXISTS MODEL_CATALOG (
    PARTITION_ID VARCHAR(50) PRIMARY KEY,
    MODEL_VERSION VARCHAR(50),
    STAGE_PATH VARCHAR(500),
    TRAINED_AT TIMESTAMP_NTZ,
    TRAIN_ROWS INTEGER,
    IS_ACTIVE BOOLEAN DEFAULT TRUE,
    FEATURE_IMPORTANCES VARIANT,
    HYPERPARAMETERS VARIANT,
    TRAIN_METRICS VARIANT
)
""").collect()

session.sql("""
CREATE TABLE IF NOT EXISTS MODEL_CATALOG_HISTORY (
    PARTITION_ID VARCHAR(50) PRIMARY KEY,
    MODEL_VERSION VARCHAR(50),
    STAGE_PATH VARCHAR(500),
    TRAINED_AT TIMESTAMP_NTZ,
    TRAIN_ROWS INTEGER,
    IS_ACTIVE BOOLEAN DEFAULT TRUE,
    FEATURE_IMPORTANCES VARIANT,
    HYPERPARAMETERS VARIANT,
    TRAIN_METRICS VARIANT,
    RETIREMENT_REASON VARCHAR(200)
)
""").collect()


print("✓ Model governance tables ready")

✓ Model governance tables ready


In [7]:
# 8. Create Output Tables
session.sql("""
CREATE TABLE IF NOT EXISTS TRAINING_METRICS (
    PARTITION_ID VARCHAR(50),
    MODEL_VERSION VARCHAR(50),
    TRAIN_ROWS INTEGER,
    TEST_ROWS INTEGER,
    TRAINED_AT VARCHAR(50),
    FEATURE_IMPORTANCES VARIANT,
    HYPERPARAMETERS VARIANT,
    TRAIN_METRICS VARIANT
)
""").collect()

session.sql("""
CREATE TABLE IF NOT EXISTS PREDICTION_TABLE (
    ACTUAL_DATE DATE,
    PARTITION_ID VARCHAR(10),
    TARGET FLOAT,
    PREDICTION FLOAT
)
""").collect()

print("✓ Output tables ready")

✓ Output tables ready


In [10]:
# 9. Configure warehouse
session.sql(f"""
    CREATE WAREHOUSE IF NOT EXISTS {WAREHOUSE}
    WITH
        WAREHOUSE_SIZE = MEDIUM
        MIN_CLUSTER_COUNT = 1
        MAX_CLUSTER_COUNT = 16
        SCALING_POLICY = 'STANDARD'; 
""").collect()

if COMPUTE_POOL in [r.name for r in session.sql("SHOW COMPUTE POOLS;").collect()]:
    print(f"Compute pool {COMPUTE_POOL} already exists:")
    session.sql(f"DESCRIBE COMPUTE POOL {COMPUTE_POOL};").show()
else:
    session.sql(f"""
        CREATE COMPUTE POOL {COMPUTE_POOL}
        WITH
            MIN_NODES = 1
            MAX_NODES = {TARGET_INSTANCES}
            INSTANCE_FAMILY = CPU_X64_S
            AUTO_SUSPEND_SECS = 10;
    """).collect()

print(f"✓ Warehouse and compute pool ready.")

Compute pool MMT_POC_POOL_50 already exists:
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"name"           |"state"    |"min_nodes"  |"max_nodes"  |"instance_family"  |"num_services"  |"num_jobs"  |"auto_suspend_secs"  |"auto_resume"  |"active_nodes"  |"idle_nodes"  |"target_nodes"  |"created_on"                      |"resumed_on"                      |"updated_on"                      |"owner"      |"comment"  |"is_exclusive"  |"application"  |"error_code"  |"status_message"  |"placement_group"  |
-----------------------------------------------------------------------------------------------------------------------------

In [11]:
# 10. Verify Setup
print("\n" + "="*60)
print("INFRASTRUCTURE SETUP COMPLETE")
print("="*60)

tables = session.sql("SHOW TABLES IN SCHEMA FORECASTING").collect()
print(f"\nTables created: {len(tables)}")
for t in tables:
    print(f"  • {t['name']}")

stages = session.sql("SHOW STAGES IN SCHEMA FORECASTING").collect()
print(f"\nStages created: {len(stages)}")
for s in stages:
    print(f"  • @{s['name']}")

print(f"\n✓ Ready to run 01_poc_data.ipynb or connect to real EDW data")


INFRASTRUCTURE SETUP COMPLETE

Tables created: 5
  • FEATURE_TABLE
  • MODEL_CATALOG
  • MODEL_CATALOG_HISTORY
  • PREDICTION_TABLE
  • TRAINING_METRICS

Stages created: 1
  • @ML_STAGE

✓ Ready to run 01_poc_data.ipynb or connect to real EDW data
